In [0]:
import sys
sys.path.append('./..')

from config.tables.table_config import TABLE_CONFIG
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.table(TABLE_CONFIG.DATA_PROCESSING)

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

#-----------------------------------------
# Encode gender
#-----------------------------------------

# Manual gender encoding to avoid Spark Connect model size limit
df = df.withColumn(
    "gender_idx",
    F.when(F.col("gender") == "F", 0.0)
     .when(F.col("gender") == "M", 1.0)
     .when(F.col("gender") == "O", 2.0)
)

#-----------------------------------------
# Features
#-----------------------------------------

feature_cols = [
    "age",
    "gender_idx",
    "credit_card_limit",
    "registered_on_days",
    "total_transactions",
    "avg_ticket",
    "max_spent",
    "min_value",
    "duration",
    "discount_value",
    "has_web",
    "has_email",
    "has_mobile",
    "has_social"
]

df = df.fillna(0, subset=["has_web", "has_email", "has_mobile", "has_social", "discount_value", "duration", "min_value"])
df = df.fillna('no_offer', ['offer_type'])

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

df = assembler.transform(df)

In [0]:
models = {}

for offer in ['informational', 'bogo', 'discount', 'no_offer']:

    offer_df = df.filter(
        F.col("offer_type") == offer
    )

    train_df, test_df = offer_df.randomSplit([0.8, 0.2], seed=42)

    train_df.write.mode("overwrite").saveAsTable(f"ifood_case.default.train_{offer}")
    test_df.write.mode("overwrite").saveAsTable(f"ifood_case.default.test_{offer}")

    model = RandomForestRegressor(
        featuresCol="features",
        labelCol="total_spent",
        numTrees=100,
        maxDepth=15
    ).fit(train_df)

    models[offer] = model

In [0]:
from pyspark.sql.functions import abs

for offer in ['informational', 'bogo', 'discount', 'no_offer']:

    test = spark.table(f'ifood_case.default.test_{offer}')
    test = models[offer].transform(test)
    test = test.withColumn("abs_error", abs(F.col("total_spent") - F.col("prediction")))

    mae = test.agg(F.avg("abs_error").alias("MAE")).select("MAE").collect()[0]['MAE']
    print(f'{offer} - MAE {mae}')

In [0]:
prediction_df = df.select("account_id").drop_duplicates()
unique_client_df = df.drop_duplicates(['account_id'])
for offer_name, model in models.items():
    pred = (
        model.transform(unique_client_df)
        .select(
            "account_id",
            F.col("prediction").alias(
                f"pred_{offer_name}"
            )
        )
    )

    prediction_df = prediction_df.join(
        pred,
        on="account_id"
    )

In [0]:
prediction_cols = [
    "pred_bogo",
    "pred_discount",
    "pred_informational",
    "pred_no_offer"
]

prediction_df = prediction_df.withColumn(
    "predictions",
    F.array(*[F.col(c) for c in prediction_cols])
)

prediction_df = prediction_df.withColumn(
    "max_prediction",
    F.array_max("predictions")
)

In [0]:
prediction_df = prediction_df.withColumn(
    "recommended_offer",
    F.when(
        F.col("max_prediction") == F.col("pred_bogo"),
        "bogo"
    )
    .when(
        F.col("max_prediction") == F.col("pred_discount"),
        "discount"
    )
    .when(
        F.col("max_prediction") == F.col("pred_informational"),
        "informational"
    )
    .otherwise("no_offer")
)

In [0]:
random_test = transactions_extracted.where('reward is not null').groupby('account_id').agg(F.sum('reward').alias('reward')).sample(.20, seed=42)

In [0]:
(random_test.join(prediction_df.where('recommended_offer = "no_offer"'),
                 on=['account_id'], how='left')
 .select(F.sum('reward').alias('reward'), 
         F.sum((F.when(F.col('pred_informational').isNull(), F.lit(0)).otherwise(F.col('reward')))).alias('not_necessary'))
 .select(F.col('not_necessary'), F.col('reward'), 100*F.col('not_necessary')/(F.col('reward')))
 .display())

In [0]:
transactions_raw.where('event = "transaction"').groupby('account_id').agg(F.sum('amount').alias('amount')).agg(F.avg('amount')).display()

In [0]:
prediction_df.agg(F.avg('max_prediction')).display()

In [0]:
prediction_df.select(
    "account_id",
    "recommended_offer",
    "max_prediction"
).display()


In [0]:
df_uplift = df \
    .withColumn("treatment", F.when(F.col("cnt_viewed") > 0, 1).otherwise(0)) \
    .withColumn("target", F.col("is_converted_valid"))

# Visualizar distribuição das turmas de tratamento/controle
df_uplift.groupBy("treatment", "target").count().show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

In [0]:
# Mapeando variáveis
num_cols = [
    'age', 'credit_card_limit', 
    #'account_age_days', 
    'total_transactions', 
    'total_spent', 'avg_ticket', 'max_spent', 
    # 'min_value', 'duration', 
    # 'discount_value', 
    # 'has_web', 'has_email', 'has_mobile', 'has_social'
]
cat_cols = ['gender']

# Indexer & OneHotEncoder para variáveis categóricas
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec") for c in cat_cols]

# Vetorizador de Features
assembler_inputs = num_cols + [f"{c}_vec" for c in cat_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

# Pipeline de Transformação
feature_pipeline = Pipeline(stages=indexers + encoders + [assembler])
feature_model = feature_pipeline.fit(df_uplift)
df_transformed = feature_model.transform(df_uplift)

# Split Treino (80%) e Teste (20%)
train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)

In [0]:
train_m0 = train_df.filter(F.col("treatment") == 0)
train_m0.display()

In [0]:
# Separar os grupos de Tratamento (W=1) e Controle (W=0) no treino
train_m1 = train_df.filter(F.col("treatment") == 1)
train_m0 = train_df.filter(F.col("treatment") == 0)

# Instanciar os classificadores GBT
gbt_m1 = GBTClassifier(featuresCol="features", labelCol="target", maxDepth=5, maxIter=100, seed=42)
gbt_m0 = GBTClassifier(featuresCol="features", labelCol="target", maxDepth=5, maxIter=100, seed=42)

# Treinar Modelo M1 (Grupo que recebeu/viu a oferta)
print("Treinando Modelo de Tratamento (M1)...")
model_m1 = gbt_m1.fit(train_m1)

# Treinar Modelo M0 (Grupo de controle/sem oferta)
print("Treinando Modelo de Controle (M0)...")
model_m0 = gbt_m0.fit(train_m0)

In [0]:
from pyspark.sql.types import DoubleType

# UDF para extrair a probabilidade da classe positiva (P(Y=1))
# get_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())

# 1. Obter predições do M1 no conjunto de Teste
preds_m1 = model_m1.transform(test_df) \
    .withColumn("prob_with_coupon", F.col("probability")) \
    .select("account_id", "offer_id", "prob_with_coupon")

# 2. Obter predições do M0 no conjunto de Teste
preds_m0 = model_m0.transform(test_df) \
    .withColumn("prob_without_coupon", F.col("probability")) \
    .select("account_id", "offer_id", "prob_without_coupon")

# 3. Consolidar os resultados e calcular o Uplift Score
uplift_results = test_df \
    .join(preds_m1, on=["account_id", "offer_id"]) \
    .join(preds_m0, on=["account_id", "offer_id"]) \
    .withColumn("uplift_score", F.round(F.col("prob_with_coupon") - F.col("prob_without_coupon"), 3))

uplift_results.select("account_id", "prob_with_coupon", "prob_without_coupon", "uplift_score").sort("uplift_score", ascending=False).drop_duplicates().display()

In [0]:
uplift_results.select("account_id", "prob_with_coupon", "prob_without_coupon", "uplift_score").sort("prob_without_coupon", ascending=True).drop_duplicates().display()

In [0]:
# Categorizar cada cliente/oferta nos quadrantes de Uplift
df_segmented = uplift_results.withColumn(
    "uplift_segment",
    F.when(F.col("uplift_score") > 0.15, "Persuadable (ENVIAR CUPOM)")
     .when(F.col("prob_without_coupon") > 0.60, "Sure Thing (NÃO ENVIAR - Compraria Organicamente)")
     .when(F.col("prob_with_coupon") < 0.10, "Lost Cause (NÃO ENVIAR - Não Converte)")
     .otherwise("Neutro")
)

# Resumo executivo dos segmentos encontrados
df_segmented.groupBy("uplift_segment").count().orderBy(F.col("count").desc()).show(truncate=False)

In [0]:
from pyspark.sql.types import DoubleType

# UDF para extrair a probabilidade da classe positiva (P(Y=1))
get_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())

# 1. Obter predições do M1 no conjunto de Teste
preds_m1 = model_m1.transform(test_df) \
    .withColumn("prob_with_coupon", get_prob_1(F.col("probability"))) \
    .select("account_id", "offer_id", "prob_with_coupon")

# 2. Obter predições do M0 no conjunto de Teste
preds_m0 = model_m0.transform(test_df) \
    .withColumn("prob_without_coupon", get_prob_1(F.col("probability"))) \
    .select("account_id", "offer_id", "prob_without_coupon")

# 3. Consolidar os resultados e calcular o Uplift Score
uplift_results = test_df \
    .join(preds_m1, on=["account_id", "offer_id"]) \
    .join(preds_m0, on=["account_id", "offer_id"]) \
    .withColumn("uplift_score", F.col("prob_with_coupon") - F.col("prob_without_coupon"))

uplift_results.select("account_id", "offer_id", "prob_with_coupon", "prob_without_coupon", "uplift_score").show(10)

In [0]:
from pyspark.sql import functions as F

transaction_features = (
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count(F.when(F.col("event") == "transaction", 1)).alias("transaction_frequency"),
        F.coalesce(
            F.sum(
                F.when(F.col("event") == "transaction", F.col("amount"))
                .otherwise(0)
            ),
            F.lit(0)
        ).alias("total_spend"),
        F.avg(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("avg_transaction_value"),
        F.stddev(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("transaction_value_std"),
        F.max(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("max_transaction"),
        F.min(
            F.when(F.col("event") == "transaction", F.col("amount"))
        ).alias("min_transaction")
    )
)

transaction_features.display()

In [0]:
from pyspark.sql import functions as F

offer_features = (
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count(
            F.when(F.col("event") == "offer received", 1)
        ).alias("offers_received_count"),

        F.count(
            F.when(F.col("event") == "offer viewed", 1)
        ).alias("offers_viewed_count"),

        F.count(
            F.when(F.col("event") == "offer completed", 1)
        ).alias("offers_completed_count")
    )
    .withColumn(
        "view_rate",
        F.when(
            F.col("offers_received_count") > 0,
            F.col("offers_viewed_count") / F.col("offers_received_count")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "completion_rate_after_view",
        F.when(
            F.col("offers_viewed_count") > 0,
            F.col("offers_completed_count") / F.col("offers_viewed_count")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "overall_completion_rate",
        F.when(
            F.col("offers_received_count") > 0,
            F.col("offers_completed_count") / F.col("offers_received_count")
        ).otherwise(F.lit(0.0))
    )
)

offer_features.display()

In [0]:
offer_features.join(transaction_features, on=['account_id']).display()

In [0]:
offer_events = transactions_df \
    .filter(F.col("offer_id").isNotNull()) \
    .groupBy("account_id", "offer_id") \
    .agg(
        F.sum(F.when(F.col("event") == "offer received", 1).otherwise(0)).alias("cnt_received"),
        F.sum(F.when(F.col("event") == "offer viewed", 1).otherwise(0)).alias("cnt_viewed"),
        F.sum(F.when(F.col("event") == "offer completed", 1).otherwise(0)).alias("cnt_completed")
    ) \
    .withColumn("is_converted_valid", 
        F.when((F.col("cnt_viewed") > 0) & (F.col("cnt_completed") > 0), 1).otherwise(0)
    )

In [0]:
user_spend_stats = transactions_df \
    .filter(F.col("event") == "transaction") \
    .groupBy("account_id") \
    .agg(
        F.count("amount").alias("total_transactions"),
        F.sum("amount").alias("total_spent"),
        F.avg("amount").alias("avg_ticket"),
        F.max("amount").alias("max_spent")
    )

In [0]:
df.select('is_converted_valid').display()